### Celda 1 — Config

In [ ]:
# ============================================================
# NB_TRNSF_GoldVentas_FactVentas
# Capa: Gold | Modelo: ventas | Tabla: f_ventas
# Fuentes: lh_silver.ADVWDB.SalesOrderDetail
#          lh_silver.SFTP.SalesOrderHeader
# ============================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# IDs de workspace y lakehouses — SIN hardcodear (TP)
# lakehouse.get() funciona en pipeline y en ejecución manual.
# El pipeline setea lh_gold como defaultLakehouse de los notebooks Gold.
_lh_gold   = notebookutils.lakehouse.get('lh_gold')
_lh_silver = notebookutils.lakehouse.get('lh_silver')
WS_ID      = _lh_gold.workspaceId
ART_GOLD   = _lh_gold.id
ART_SILVER = _lh_silver.id

# Tablas origen — construidas dinámicamente desde config (TP: no hardcodear)
_cfg = spark.sql("""
    SELECT origen, nombre_tabla
    FROM lh_config.dbo.origin_bronze_silver
    WHERE activo = 1
""").collect()
_cfg_map = {(r['origen'], r['nombre_tabla']): r for r in _cfg}

_sod_row = _cfg_map.get(('ADVWDB', 'SalesOrderDetail'))
_soh_row = _cfg_map.get(('SFTP',   'SalesOrderHeader'))
if not _sod_row or not _soh_row:
    raise Exception("Config incompleta para SalesOrderDetail o SalesOrderHeader")

tabla_sod  = f"lh_silver.{_sod_row['origen']}.{_sod_row['nombre_tabla']}"
tabla_soh  = f"lh_silver.{_soh_row['origen']}.{_soh_row['nombre_tabla']}"

# FIX D01/D02: leer modelo y nombre_tabla Gold desde origin_gold
_og_rows = spark.sql("""
    SELECT nombre_notebook, modelo, nombre_tabla
    FROM lh_config.dbo.origin_gold
    WHERE activo = 1
""").collect()
_og_map = {r['nombre_notebook']: r for r in _og_rows}

_og_fact = _og_map.get('NB_TRNSF_GoldVentas_FactVentas')
_og_cal  = _og_map.get('NB_TRNSF_GoldVentas_DimCalendario')
_og_prod = _og_map.get('NB_TRNSF_GoldVentas_DimProducto')
if not _og_fact or not _og_cal or not _og_prod:
    raise Exception("Config incompleta en origin_gold — verificar NB_TRNSF_Gold* activos")

tabla_fact = f"lh_gold.{_og_fact['modelo']}.{_og_fact['nombre_tabla']}"  # 3-part para saveAsTable/DROP TABLE
tabla_cal  = f"lh_gold.{_og_cal['modelo']}.{_og_cal['nombre_tabla']}"    # 3-part para saveAsTable/DROP TABLE
tabla_prod = f"lh_gold.{_og_prod['modelo']}.{_og_prod['nombre_tabla']}"  # 3-part para saveAsTable/DROP TABLE

# Variables 1-part para queries SQL — spark_catalog no acepta 3-part en SQL inline
sql_fact  = _og_fact['nombre_tabla']   # f_ventas
sql_cal   = _og_cal['nombre_tabla']    # d_calendario
sql_prod  = _og_prod['nombre_tabla']   # d_producto
sql_schema = _og_fact['modelo']        # ventas

# FIX F09: abfs_fact construido desde las mismas variables (consistencia ABFS/catalog)
abfs_fact = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_GOLD}/Tables/{_og_fact['modelo']}/{_og_fact['nombre_tabla']}"
)

print("✅ Paths y variables configurados desde origin_gold + origin_bronze_silver")
print(f"   tabla_sod  : {tabla_sod}")
print(f"   tabla_soh  : {tabla_soh}")
print(f"   tabla_fact : {tabla_fact}")
print(f"   tabla_cal  : {tabla_cal}")
print(f"   tabla_prod : {tabla_prod}")
print(f"   abfs_fact  : {abfs_fact}")


### Celda 2 — Join Silver → Fact

In [ ]:
# ============================================================
# Leer Silver + construir df_fact (JOIN SalesOrderDetail + Header)
# LIMIT 1000 obligatorio durante desarrollo (TP)
# ============================================================
df_sod = spark.sql(f"SELECT * FROM {tabla_sod} LIMIT 1000")
df_soh = spark.sql(f"SELECT * FROM {tabla_soh} LIMIT 1000")
print(f"SalesOrderDetail : {df_sod.count()} filas")
print(f"SalesOrderHeader : {df_soh.count()} filas")

# JOIN — construye df_fact para el MERGE de la siguiente celda
df_fact = (
    df_sod.alias('sod')
    .join(df_soh.alias('soh'), 'SalesOrderID', 'inner')
    .select(
        F.col('sod.SalesOrderDetailID').alias('id_detalle'),
        F.col('sod.SalesOrderID').alias('id_orden'),
        F.col('sod.ProductID').alias('id_producto'),
        F.col('soh.CustomerID').alias('id_cliente'),
        F.col('soh.OrderDate').alias('fecha_orden'),
        F.col('soh.TerritoryID').alias('id_territorio'),
        F.col('sod.OrderQty').alias('cantidad'),
        F.col('sod.UnitPrice').alias('precio_unitario'),
        F.col('sod.UnitPriceDiscount').alias('descuento'),
        F.col('sod.LineTotal').alias('total_linea'),
        F.col('soh.TotalDue').alias('total_orden'),
        F.col('soh.OnlineOrderFlag').alias('es_online'),
        F.col('sod.ModifiedDate').alias('fecha_modificacion')
    )
)
print(f"✅ df_fact construido: {df_fact.count()} filas")
display(df_fact.limit(5))


### Celda 3 — Guardar con MERGE en Gold

In [ ]:
# ============================================================
# Guardar f_ventas en Gold
# tabla_fact = lh_gold.{modelo}.{nombre} — 3-part dinámico desde config
# DROP TABLE con 3-part nunca lanza AnalysisException
# Después del write: USE lh_gold.ventas para queries analíticas (1-part)
# ============================================================

# Limpiar tabla anterior
spark.sql(f"DROP TABLE IF EXISTS {tabla_fact}")
print(f"🗑️  DROP TABLE IF EXISTS {tabla_fact}")

# Escribir y registrar en metastore
(df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabla_fact))

# Setear contexto SQL para que las queries analíticas resuelvan 1-part names
spark.sql(f"USE lh_gold.{sql_schema}")
print(f"✅ f_ventas guardada: {df_fact.count()} filas → {tabla_fact}")
print(f"   contexto SQL activo: lh_gold.{sql_schema}")


### Celda 4 — Analítica: Top Clientes semana actual vs anterior

In [ ]:
# ============================================================
# Lógica de negocio: Top 10 clientes
# semana actual vs semana anterior
# Según ejemplo del TP: "Top Clientes de última semana
# en comparación a la semana pasada"
# ============================================================

# FIX F09: usar tabla_fact (catalog name) en lugar de delta.`abfs_fact`
# FIX F10: condición semana_anterior acotada con AND fecha_orden < inicio_semana_actual
#           para evitar solapamiento entre semana_actual y semana_anterior
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_top_clientes AS
    WITH semanas AS (
        SELECT
            date_trunc('week', current_date())          AS inicio_semana_actual,
            date_add(date_trunc('week', current_date()), -7) AS inicio_semana_anterior
    ),
    ventas_por_semana AS (
        SELECT
            f.id_cliente,
            CASE
                WHEN f.fecha_orden >= s.inicio_semana_actual
                    THEN 'semana_actual'
                WHEN f.fecha_orden >= s.inicio_semana_anterior
                    AND f.fecha_orden <  s.inicio_semana_actual
                    THEN 'semana_anterior'
            END AS semana,
            SUM(f.total_linea)        AS total_ventas,
            COUNT(DISTINCT f.id_orden) AS cant_ordenes
        FROM {sql_fact} f
        CROSS JOIN semanas s
        WHERE f.fecha_orden >= s.inicio_semana_anterior
        GROUP BY f.id_cliente, semana
    )
    SELECT
        id_cliente,
        MAX(CASE WHEN semana = 'semana_actual'   THEN total_ventas  ELSE 0 END) AS ventas_semana_actual,
        MAX(CASE WHEN semana = 'semana_anterior' THEN total_ventas  ELSE 0 END) AS ventas_semana_anterior,
        MAX(CASE WHEN semana = 'semana_actual'   THEN cant_ordenes  ELSE 0 END) AS ordenes_semana_actual,
        MAX(CASE WHEN semana = 'semana_anterior' THEN cant_ordenes  ELSE 0 END) AS ordenes_semana_anterior
    FROM ventas_por_semana
    GROUP BY id_cliente
    ORDER BY ventas_semana_actual DESC
    LIMIT 10
""")

display(spark.sql("SELECT * FROM vw_top_clientes"))


### Celda 5 — Analítica: Ventas por producto y mes

In [ ]:
# ============================================================
# Lógica adicional: Ventas agregadas por producto y mes
# Joins con dimensiones Gold
# ============================================================

# FIX F09: usar tabla_fact (catalog name) — consistente con tabla_cal y tabla_prod
display(spark.sql(f"""
    SELECT
        c.CodTiempo                          AS periodo,
        c.Mes_Year                           AS mes_anio,
        p.nombre_producto,
        p.linea_producto,
        COUNT(DISTINCT f.id_orden)           AS cant_ordenes,
        SUM(f.cantidad)                      AS unidades_vendidas,
        ROUND(SUM(f.total_linea), 2)         AS total_ventas,
        ROUND(AVG(f.precio_unitario), 2)     AS precio_promedio
    FROM {sql_fact} f
    JOIN {sql_cal}  c ON f.fecha_orden = c.Fecha
    JOIN {sql_prod} p ON f.id_producto  = p.id_producto
    GROUP BY c.CodTiempo, c.Mes_Year, p.nombre_producto, p.linea_producto
    ORDER BY c.CodTiempo DESC, total_ventas DESC
    LIMIT 1000
"""))
